In [1]:
import argparse
import os
import pickle

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
exp_name = 'collision-walking-rand'  # ckpt = 4000
ckpt = 200

action_scale = 1.0 # 動作のスケールを調整

In [4]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [5]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [6]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.01
# env_cfg['substeps'] = 10
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 100
env_cfg['termination_if_pitch_greater_than'] = 100

In [7]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [8]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

print("env_reset:", obs["policy"])

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
env_reset: tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
    

In [9]:
with torch.no_grad():
    
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    cnt += 1


Original actions :  tensor([[-0.1029,  0.0038,  0.8968, -0.2168, -0.4307, -0.8971,  0.3350,  0.0641,
          0.6694, -0.0840, -1.4133,  0.1074]], device='cuda:0')
Scaled actions :  tensor([[-0.1029,  0.0038,  0.8968, -0.2168, -0.4307, -0.8971,  0.3350,  0.0641,
          0.6694, -0.0840, -1.4133,  0.1074]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


tensor([[-3.3750e-06, -7.5798e-03, -1.3292e-06, -1.6181e-08,  2.5237e-19,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.5636e-08,
          2.7593e-08, -1.4377e-04,  3.6383e-04, -1.9670e-04, -8.4460e-08,
         -8.4522e-08, -4.0163e-08, -1.4383e-04,  3.6383e-04, -1.9675e-04,
          6.5024e-07,  1.7818e-06,  1.3796e-06, -7.1954e-03,  1.8185e-02,
         -9.4705e-03, -4.2230e-06, -4.2261e-06, -2.0081e-06, -7.1981e-03,
          1.8190e-02, -9.4732e-03,  3.2512e-05, -1.0286e-01,  3.8334e-03,
          8.9682e-01, -2.1678e-01, -4.3067e-01, -8.9713e-01,  3.3501e-01,
          6.4110e-02,  6.6945e-01, -8.4019e-02, -1.4133e+00,  1.0743e-01]],
       device='cuda:0')


In [10]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[-0.0054, -0.1901,  1.2294, -0.2824, -1.2842, -1.2820,  0.7636,  0.2681,
          0.9647, -0.3552, -2.6657,  0.0061]], device='cuda:0')
Scaled actions :  tensor([[-0.0054, -0.1901,  1.2294, -0.2824, -1.2842, -1.2820,  0.7636,  0.2681,
          0.9647, -0.3552, -2.6657,  0.0061]], device='cuda:0')
tensor([[-2.6989e-02, -1.6594e-01, -5.4054e-02, -5.9021e-03, -3.1667e-04,
         -9.9998e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.7782e-03,
          5.3923e-05,  9.3827e-03,  1.4707e-03, -6.0442e-03, -1.9990e-03,
          3.6719e-03,  8.5624e-06,  9.2516e-03,  3.8456e-03, -2.0968e-02,
          8.6968e-03, -3.8501e-02,  1.7867e-02,  4.7706e-02,  5.2658e-03,
          1.0053e-01, -8.6656e-02,  1.1149e-01,  6.8894e-03,  5.7377e-02,
          2.5443e-02, -1.7660e-01,  7.6864e-02, -5.3788e-03, -1.9009e-01,
          1.2294e+00, -2.8242e-01, -1.2842e+00, -1.2820e+00,  7.6357e-01,
          2.6806e-01,  9.6466e-01, -3.5523e-01, -2.6657e+00,  6.1060e-03]],
    

In [11]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[-0.0536, -0.2215,  1.3429, -0.3862, -1.3808, -1.1916,  0.7176,  0.2994,
          1.0412, -0.4102, -2.5836, -0.1137]], device='cuda:0')
Scaled actions :  tensor([[-0.0536, -0.2215,  1.3429, -0.3862, -1.3808, -1.1916,  0.7176,  0.2994,
          1.0412, -0.4102, -2.5836, -0.1137]], device='cuda:0')
tensor([[ 4.2839e-02, -9.4862e-02, -1.4341e-01, -1.0763e-02, -1.6703e-03,
         -9.9994e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.8140e-03,
         -5.8250e-04,  1.9008e-02,  1.9811e-03, -1.8219e-02, -1.2130e-02,
          1.2990e-02,  2.1597e-03,  2.0057e-02,  4.3458e-03, -3.9557e-02,
          3.8244e-03,  3.0321e-02, -1.2085e-02,  5.5045e-02, -1.7115e-02,
         -1.3846e-02, -1.2272e-02,  1.4031e-02,  1.5337e-02,  4.5593e-02,
         -1.8342e-02, -6.6337e-02, -5.7416e-02, -5.3631e-02, -2.2150e-01,
          1.3429e+00, -3.8615e-01, -1.3808e+00, -1.1916e+00,  7.1758e-01,
          2.9943e-01,  1.0412e+00, -4.1025e-01, -2.5836e+00, -1.1367e-01]],
    

In [12]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[-0.0933, -0.1103,  1.7477, -0.2924, -1.5296, -1.0542,  0.3062,  0.3921,
          0.4179, -0.0131, -2.0406,  0.0829]], device='cuda:0')
Scaled actions :  tensor([[-0.0933, -0.1103,  1.7477, -0.2924, -1.5296, -1.0542,  0.3062,  0.3921,
          0.4179, -0.0131, -2.0406,  0.0829]], device='cuda:0')
tensor([[ 4.1933e-02, -1.1710e-01, -2.0347e-01, -1.6030e-02, -2.7384e-03,
         -9.9987e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.2259e-03,
         -1.5998e-03,  3.1799e-02, -1.1044e-03, -2.4162e-02, -1.6919e-02,
          1.8778e-02,  3.2598e-03,  3.1503e-02,  3.2369e-03, -5.5720e-02,
         -4.0190e-03,  7.0271e-02, -1.6720e-02,  8.0317e-02, -5.5177e-02,
          6.1956e-03, -1.6569e-03,  8.1070e-03,  1.4346e-03,  5.3778e-02,
         -1.4931e-02, -3.8387e-02,  1.1027e-02, -9.3345e-02, -1.1026e-01,
          1.7477e+00, -2.9237e-01, -1.5296e+00, -1.0542e+00,  3.0616e-01,
          3.9208e-01,  4.1790e-01, -1.3059e-02, -2.0406e+00,  8.2916e-02]],
    

In [13]:
with torch.no_grad(): 
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    
    print(obs["policy"])
    print("step :", cnt)   
    cnt += 1


Original actions :  tensor([[-0.2216, -0.0188,  1.6854, -0.1568, -1.6929, -1.0510,  0.1842,  0.4440,
          0.2471,  0.1381, -1.5348,  0.2118]], device='cuda:0')
Scaled actions :  tensor([[-0.2216, -0.0188,  1.6854, -0.1568, -1.6929, -1.0510,  0.1842,  0.4440,
          0.2471,  0.1381, -1.5348,  0.2118]], device='cuda:0')
tensor([[ 7.9315e-03, -7.8500e-02, -2.8033e-01, -2.2964e-02, -3.8465e-03,
         -9.9973e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  4.6450e-03,
         -2.2599e-03,  4.9403e-02, -3.4182e-03, -4.1168e-02, -2.7868e-02,
          2.4489e-02,  6.0897e-03,  4.4860e-02,  4.4773e-03, -8.2317e-02,
         -8.8319e-03,  2.0098e-02, -6.4524e-04,  7.0921e-02, -1.6919e-02,
         -1.2784e-01, -9.6122e-02,  4.3696e-02,  2.0895e-02,  1.8514e-02,
          2.0601e-03, -5.4690e-02, -6.9444e-02, -2.2165e-01, -1.8815e-02,
          1.6854e+00, -1.5682e-01, -1.6929e+00, -1.0510e+00,  1.8418e-01,
          4.4397e-01,  2.4705e-01,  1.3806e-01, -1.5348e+00,  2.1180e-01]],
    

In [14]:
num_steps = 1000
for i in range(num_steps):
    with torch.no_grad():
        if i % 100 == 0:
            print("cnt :", cnt)   
        actions = policy(obs)

        # アクションに倍率を適用して動きを制限
        scaled_actions = actions * action_scale
        
        if i % 100 == 0:
            print("Original actions : ", actions)
            print("Scaled actions : ", scaled_actions)

        obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用

        cnt += 1

cnt : 5
Original actions :  tensor([[-0.2876,  0.0994,  1.7890, -0.0674, -1.2704, -0.6524, -0.0289,  0.5390,
         -0.2793, -0.0303, -0.9121,  0.4687]], device='cuda:0')
Scaled actions :  tensor([[-0.2876,  0.0994,  1.7890, -0.0674, -1.2704, -0.6524, -0.0289,  0.5390,
         -0.2793, -0.0303, -0.9121,  0.4687]], device='cuda:0')
cnt : 105
Original actions :  tensor([[-1.6107,  1.5895, -0.0911, -0.6312,  0.8768,  0.1401,  1.7189,  0.6487,
          4.2884, -1.0216, -2.8832,  1.1640]], device='cuda:0')
Scaled actions :  tensor([[-1.6107,  1.5895, -0.0911, -0.6312,  0.8768,  0.1401,  1.7189,  0.6487,
          4.2884, -1.0216, -2.8832,  1.1640]], device='cuda:0')
cnt : 205
Original actions :  tensor([[-1.5409, -0.1872, -1.1156, -2.7654,  0.3659, -0.5645, -1.1757,  0.0483,
          5.1152, -0.9540,  0.3318,  0.8190]], device='cuda:0')
Scaled actions :  tensor([[-1.5409, -0.1872, -1.1156, -2.7654,  0.3659, -0.5645, -1.1757,  0.0483,
          5.1152, -0.9540,  0.3318,  0.8190]], devic

In [15]:
env.sim.stop()